## Imports

In [1]:
import pandas as pd
import csv
import sys
from pathlib import Path
import requests
from bs4 import BeautifulSoup

In [16]:
FULL_TEXT_PATH = Path("..") / "dataset" / "text_list.csv"
BASE_URL = 'https://hudoc.echr.coe.int/app/conversion/docx/html/body?library=ECHR&id='
FOE_ECHR_PATH = Path("..") / "dataset" / "foe_echr.csv"


## Reading the dataset

Read the unprocessed json of the unstructured cases downloaded from the ECHR OD website

In [2]:
dataset_path = ".." / Path.cwd().parent / "dataset" / "echr_2_0_0_unstructured_cases.json"
df = pd.read_json(dataset_path)

## Functions

Function to extract the relevant sections of the case into markdown format

In [3]:
def extract_section_markdown(case_data:dict, section_name:str) -> str:
    """
    Extracts a specific section from the case data and formats it as markdown.

    Args:
        case_data (dict): The dictionary containing case data.
        section_name (str): The name of the section to extract (e.g., 'law', 'facts').

    Returns:
        str: Formatted markdown text of the specified section.
    """
    
    section_text = ""
    if 'content' in case_data:
        documents = case_data['content']
        for key, sections in documents.items():
            for section in sections:
                if section.get('section_name', '').upper() == section_name.upper():
                    # Append the section title with a heading format
                    section_text += f"### {section['content']}\n\n"
                    section_text += extract_elements(section['elements'])
                    section_text += "\n"

    return section_text.strip()

def extract_elements(elements, indent=0):
    content_text = ""
    for element in elements:
        # Add indentation for each level of nesting
        prefix = "  " * indent
        content_text += f"{prefix}- {element['content']}\n"
        if 'elements' in element:
            # Recursively add further nested elements with increased indentation
            content_text += extract_elements(element['elements'], indent + 1)
    return content_text

Applying the function to populate law, facts, and conclusion columns

In [4]:
df['law'] = df.apply(lambda x: extract_section_markdown(x, 'law'), axis = 1)
df['facts'] = df.apply(lambda x: extract_section_markdown(x, 'facts'), axis = 1)
df['the_conclusion'] = df.apply(lambda x: extract_section_markdown(x, 'conclusion'), axis = 1)

Function to extract html content from a webpage 

In [5]:
def get_full_text_from_html(html_text:str) -> str: 
    """
    Extracts plain text from HTML content by removing scripts, styles, and extra whitespace.

    Args:
        html_text (str): HTML content as a string.

    Returns:
        str: Cleaned plain text with unnecessary elements removed.
    """
    soup = BeautifulSoup(html_text, "html.parser")
    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    # Get plain text and replace multiple whitespace characters with a single space
    text = ' '.join(soup.get_text().split())
    return text.replace(u'\xa0', ' ')  # Replace non-breaking spaces

In [7]:
def retrieve_or_fetch_text(dataframe:pd.DataFrame, filename_path:Path, base_url:str)->list:
    """
    Retrieves text data from a CSV file if it exists; otherwise, fetches HTML text 
    from URLs constructed with item IDs in the dataframe.

    Args:
        dataframe (pd.DataFrame): DataFrame containing 'itemid' to construct URLs if CSV is absent.
        filename_path (Path): Path to the CSV file.
        base_url (str): Base URL for constructing the full URL to fetch HTML content.

    Returns:
        list: List of plain text extracted either from the CSV file or from HTML fetched from URLs.
    """
    text_list = []

    # Check if the CSV file exists
    if filename_path.exists():
        # Read text from CSV file
        csv.field_size_limit(sys.maxsize)  # Expand field size for large text
        with open(filename_path, 'r', newline='', encoding='utf-8') as file:
            reader = csv.reader(file)
            for row in reader:
                if row:  # Check if row is not empty
                    full_text = ' '.join(row)
                    text_list.append(full_text)
    else:
        # If the file does not exist, fetch HTML text from URLs
        for itemid in dataframe['itemid']:
            response = requests.get(f"{base_url}{itemid}", timeout=5)
            text_list.append(get_full_text_from_html(response.text))
    
    return text_list


If it not already extracted, extract the textual content from the HTML format of the echr database to get the full text

In [13]:
text_list = retrieve_or_fetch_text(df, filename_path, BASE_URL)
df['full_text'] = text_list

Extract the judgement type from the case data

In [9]:
df['judgement_type_1'] = df['documentcollectionid'].apply(lambda x: x[1])
df['judgement_type_2'] = df['documentcollectionid'].apply(lambda x: x[2])

Slice the dataset to keep only the relevant columns

In [10]:
def slice_dataset(dataset:pd.DataFrame, columns: list[str])->pd.DataFrame:
    """
    Slice the dataset into a subset depending on the names of columns provided

    Args:
        dataset (pd.DataFrame): The original dataset to be sliced
        columns (list[str]): the list of column names that will be kept

    Returns:
        pd.DataFrame: The sliced dataset
    """
    return dataset[columns]

In [14]:
COLUMN_LIST = ['itemid', 'docname','article','appno','judgementdate', 'law', 'facts', 'the_conclusion', 'full_text', 'respondent', 'judgementdate']
echr_df = slice_dataset(df,COLUMN_LIST)

Select the instances that are refering to violations of article 10 (freedom of expression)

In [15]:
foe_echr_df = echr_df[echr_df['article'].apply(lambda x: '10' in x)]
foe_echr_df = foe_echr_df.reset_index(drop=True)
foe_echr_df

,itemid,docname,article,appno,judgementdate,law,facts,the_conclusion,full_text,respondent,judgementdate
0,001-209033,CASE OF HANDZHIYSKI v. BULGARIA,"[35, 41, 10]",10783/14,06/04/2021 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLE ...,### THE FACTS\n\n- 2. The applicant was born ...,"### FOR THESE REASONS, THE COURT\n\n- Declares...",FOURTH SECTIONCASE OF HANDZHIYSKI v. BULGARIA(...,BGR,06/04/2021 00:00:00
1,001-57513,CASE OF KOSIEK v. GERMANY,[10],9704/82,28/08/1986 00:00:00,### AS TO THE LAW\n\n- I. THE GOVERNMENT’S P...,### AS TO THE FACTS\n\n- 11. Mr. Rolf Kosiek...,"### FOR THESE REASONS, THE COURT\n\n- Holds by...",COURT (PLENARY) CASE OF KOSIEK v. GERMANY (App...,DEU,28/08/1986 00:00:00
2,001-155196,CASE OF MEHDIYEV v. AZERBAIJAN,"[41, 3, 5, 35, 10]",59075/09,18/06/2015 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT\n\n- 1. Decl...",FIRST SECTION CASE OF MEHDIYEV v. AZERBAIJAN (...,AZE,18/06/2015 00:00:00
3,001-84268,CASE OF FEVZİ SAYGILI v. TURKEY,"[14, 10, 13]",74243/01,08/01/2008 00:00:00,### THE LAW\n\n- I. THE GOVERNMENT'S PRELIMIN...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF FEVZİ SAYGILI v. TURKEY...,TUR,08/01/2008 00:00:00
4,001-107591,CASE OF KILIÇ AND EREN v. TURKEY,[10],43807/07,29/11/2011 00:00:00,### THE LAW\n\n- I. ALLEGED VIOLATION OF ARTI...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT UNANIMOUSLY\n...",SECOND SECTION CASE OF KILIÇ AND EREN v. TURKE...,TUR,29/11/2011 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...
964,001-201087,CASE OF RELIGIOUS COMMUNITY OF JEHOVAH'S WITNE...,[10],52884/09,20/02/2020 00:00:00,### THE LAW\n\n- ALLEGED VIOLATION OF ARTICLEs...,### THE FACTS\n\n- THE CIRCUMSTANCES OF THE CA...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",FIFTH SECTIONCASE OF RELIGIOUS COMMUNITY OF JE...,AZE,20/02/2020 00:00:00
965,001-58032,"CASE OF X, Y AND Z v. THE UNITED KINGDOM","[8, 10, 14]",21830/93,22/04/1997 00:00:00,### AS TO THE LAW\n\n- I. ALLEGED VIOLATION ...,### AS TO THE FACTS\n\n- I. Circumstances of...,"### FOR THESE REASONS, THE COURT\n\n- 1. Hol...","COURT (GRAND CHAMBER) CASE OF X, Y AND Z v. TH...",GBR,22/04/1997 00:00:00
966,001-217373,CASE OF ZAO INFORMATSIONNOYE AGENTSTVO ROSBALT...,[10],16503/14,24/05/2022 00:00:00,### APPLICATION OF ARTICLE 41 OF THE CONVENTIO...,,### THE COURT’S ASSESSMENT\n\n- ALLEGED VIOLAT...,THIRD SECTIONCASE OF ZAO INFORMATSIONNOYE AGEN...,RUS,24/05/2022 00:00:00
967,001-186008,CASE OF ÇETİN AND GEDİK v. TURKEY,[10],29899/07;33333/08,04/09/2018 00:00:00,### THE LAW\n\n- I. JOINDER OF THE APPLICATIO...,### THE FACTS\n\n- I. THE CIRCUMSTANCES OF TH...,"### FOR THESE REASONS, THE COURT, UNANIMOUSLY,...",SECOND SECTION CASE OF ÇETİN AND GEDİK v. TURK...,TUR,04/09/2018 00:00:00


Save the dataset in csv

In [ ]:
foe_echr_df.to_csv(FOE_ECHR_PATH)